# AI for the Movement — Your First AI API Call

Every AI tool you'll build in this course talks to a language model through an API. Today you'll make that connection yourself — and see how AI can draft advocacy content, counter common arguments, and analyse movement strategy.

**Before you start: File → Save a copy in Drive**

---

### Set up your API key in Colab Secrets

1. Click the 🔑 key icon in the left sidebar
2. Click **"Add a new secret"**
3. Name: `OPENROUTER_API_KEY`
4. Value: paste your OpenRouter API key
5. Toggle **"Notebook access"** to ON

Don't have an API key yet? Get one at [openrouter.ai](https://openrouter.ai) → Settings → API Keys

In [ ]:
from google.colab import userdata

try:
    api_key = userdata.get('OPENROUTER_API_KEY')
    print("✅ API key loaded successfully!")
except Exception as e:
    print("❌ Could not load API key. Follow these steps:")
    print("   1. Click the 🔑 key icon in the left sidebar")
    print("   2. Click 'Add a new secret'")
    print("   3. Name: OPENROUTER_API_KEY")
    print("   4. Value: paste your OpenRouter API key")
    print("   5. Toggle 'Notebook access' to ON")
    print(f"\nError: {e}")

## Step 1: Draft Advocacy Content with AI

One of the most immediate uses of AI for advocacy is content generation — social media posts, email campaigns, talking points. Let's see what an AI model can do with a real advocacy prompt.

In [ ]:
import requests

url = "https://openrouter.ai/api/v1/chat/completions"
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "openai/gpt-4o-mini",
    "max_tokens": 300,
    "messages": [{"role": "user", "content": "Write a compelling social media post (under 280 characters) about the environmental impact of factory farming. Include a specific statistic. Make it shareable, not preachy."}]
}

try:
    r = requests.post(url, headers=headers, json=payload)
    if r.status_code == 401:
        print("❌ Invalid API key. Check your key at openrouter.ai/settings/keys")
    elif r.status_code == 402:
        print("❌ No credits. Add funds at openrouter.ai/settings/credits")
    elif r.status_code == 429:
        print("❌ Rate limited. Wait a moment and try again.")
    elif r.status_code != 200:
        print(f"❌ Error {r.status_code}: {r.text}")
    else:
        data = r.json()
        print(data["choices"][0]["message"]["content"])
        print("\n✅ First API call complete!")
except Exception as e:
    print(f"❌ Request failed: {e}")

## Step 2: Counter Common Arguments

Advocacy organisations deal with the same counter-arguments constantly. AI can help draft evidence-based responses at scale.

In [ ]:
payload = {
    "model": "openai/gpt-4o-mini",
    "max_tokens": 300,
    "messages": [{"role": "user", "content": "Someone says: 'But humans have always eaten meat, it's natural.' Write a brief, respectful, evidence-based response that addresses this argument without being condescending. Include one scientific reference."}]
}

try:
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 200:
        print(f"❌ Error {response.status_code}: {response.text}")
    else:
        response_data = response.json()
        print(response_data["choices"][0]["message"]["content"])
        print("\n✅ Counter-argument generated!")
except Exception as e:
    print(f"❌ Request failed: {e}")

## Step 3: Understand the Cost

Every API call costs money. Understanding token economics is essential for building sustainable advocacy tools.

A **token** is roughly 3/4 of a word. You pay for **tokens in** (your prompt) and **tokens out** (the model's response). The pricing differs between models — and between input and output.

In [ ]:
# Uses the response from Step 2
try:
    usage = response_data["usage"]
    model_name = response_data["model"]

    prompt_tokens = usage["prompt_tokens"]
    completion_tokens = usage["completion_tokens"]
    total_tokens = usage["total_tokens"]

    # GPT-4o-mini pricing: $0.15 per 1M input tokens, $0.60 per 1M output tokens
    input_cost = prompt_tokens * 0.15 / 1_000_000
    output_cost = completion_tokens * 0.60 / 1_000_000
    total_cost = input_cost + output_cost
    calls_per_dollar = int(1.0 / total_cost) if total_cost > 0 else 0

    print(f"Model:             {model_name}")
    print(f"Prompt tokens:     {prompt_tokens}")
    print(f"Completion tokens: {completion_tokens}")
    print(f"Total tokens:      {total_tokens}")
    print(f"Estimated cost:    ${total_cost:.6f}")
    print(f"\nAt this rate, $1 would buy you roughly {calls_per_dollar:,} calls like this.")
except NameError:
    print("\u274c Run Step 2 first — this cell uses that response data.")

## Step 4: Compare Models

Different models have different capabilities and costs. A cheap model might be fine for drafting social posts. Fact-checking claims that your organisation will publish? You might want a more capable model. Let's compare.

In [ ]:
import time

models = [
    {"id": "openai/gpt-4o-mini", "input": 0.15, "output": 0.60},
    {"id": "anthropic/claude-3.5-haiku", "input": 1.00, "output": 5.00},
    {"id": "anthropic/claude-sonnet-4", "input": 3.00, "output": 15.00},
]

prompt = "Analyse the following claim and rate its accuracy from 1-10 with a brief explanation: 'Animal agriculture is responsible for more greenhouse gas emissions than the entire transportation sector combined.'"

for i, model in enumerate(models):
    print(f"\n{'='*60}")
    print(f"Model: {model['id']}")
    print(f"{'='*60}")
    try:
        r = requests.post(url, headers=headers, json={
            "model": model["id"], "max_tokens": 300,
            "messages": [{"role": "user", "content": prompt}]
        })
        if r.status_code != 200:
            print(f"❌ Error {r.status_code}: {r.text[:200]}")
        else:
            d = r.json()
            print(d["choices"][0]["message"]["content"])
            u = d["usage"]
            cost = u["prompt_tokens"] * model["input"] / 1e6 + u["completion_tokens"] * model["output"] / 1e6
            print(f"\nTokens: {u['prompt_tokens']} in / {u['completion_tokens']} out | Cost: ${cost:.6f}")
    except Exception as e:
        print(f"❌ Failed: {e}")
    if i < len(models) - 1:
        time.sleep(1)

> ⚠️ **Notice the differences.** The budget model might give a quick answer. The premium model might catch nuances the others miss. For advocacy, accuracy matters — misinformation undermines credibility. Choose models based on the stakes of the task.

## Step 5: System Prompts — Giving AI a Role

So far, we've sent simple user messages. But the API supports a more powerful pattern: **system prompts**.

- **System message**: Sets the AI's identity, behaviour, and constraints
- **User message**: The actual question or task

System prompts are how you shape AI behaviour. This is the foundation of every advocacy chatbot, every content tool, every automated assistant.

In [ ]:
system_prompt = """You are VEG3, an AI assistant for animal advocates. You provide evidence-based information about animal agriculture, veganism, and advocacy strategy. You are compassionate, accurate, and never condescending. When citing statistics, note the source. When addressing counter-arguments, lead with empathy before evidence."""

user_message = "I'm having dinner with family who keep asking why I went vegan. How do I handle the conversation without starting a fight?"

try:
    r = requests.post(url, headers=headers, json={
        "model": "openai/gpt-4o-mini", "max_tokens": 300,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    })
    if r.status_code != 200:
        print(f"❌ Error {r.status_code}: {r.text}")
    else:
        print(r.json()["choices"][0]["message"]["content"])
        print("\n✅ VEG3 responded!")
except Exception as e:
    print(f"❌ Request failed: {e}")

> 💡 **This is what you're building toward.** VEG3 is a real project — an AI assistant for animal advocates that existed before ChatGPT. In this course, you'll learn to build tools like this: AI systems shaped by system prompts, powered by API calls, designed to serve the movement.

> 🎯 **Your Turn:** Try your own advocacy prompt. What would be useful for the movement?

In [ ]:
my_system_prompt = """You are VEG3, an AI assistant for animal advocates. You provide evidence-based information about animal agriculture, veganism, and advocacy strategy. You are compassionate, accurate, and never condescending. When citing statistics, note the source. When addressing counter-arguments, lead with empathy before evidence."""

my_user_prompt = "..."  # <-- Replace with your own prompt!

# Ideas to try:
# "Draft an email to a local council member about factory farm expansion"
# "Explain the connection between animal agriculture and pandemic risk"
# "Create a FAQ for a vegan outreach event"
# "Summarise the key arguments in the Cambridge Declaration on Consciousness"
# "Write a pitch for a tech tool that helps animal shelters coordinate rescues"

try:
    r = requests.post(url, headers=headers, json={
        "model": "openai/gpt-4o-mini", "max_tokens": 300,
        "messages": [
            {"role": "system", "content": my_system_prompt},
            {"role": "user", "content": my_user_prompt}
        ]
    })
    if r.status_code != 200:
        print(f"❌ Error {r.status_code}: {r.text}")
    else:
        print(r.json()["choices"][0]["message"]["content"])
        print("\n✅ Done!")
except Exception as e:
    print(f"❌ Request failed: {e}")

## Summary and Cost Control Checklist

You've just made your first AI API calls. Before you go further, commit this checklist to memory:

- ☐ Always set `max_tokens` on every API call
- ☐ Start with the cheapest model for development and testing
- ☐ Check your usage at [openrouter.ai/activity](https://openrouter.ai/activity)
- ☐ Set per-key spending limits (Settings → API Keys)
- ☐ Remember: conversation history is resent every turn — turn 10 costs 10× turn 1
- ☐ Never put API keys in code files

---

Every advocacy chatbot, every content generator, every sentiment analysis tool — they all start here. Same endpoint, same JSON, same token economics. The rest of this course is learning to build increasingly powerful tools on this foundation.